In [1]:
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool as CatBoostPool
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
from optuna.samplers import TPESampler
from tqdm.autonotebook import tqdm
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
# ==================== 评估指标 ====================

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate(y_true, y_pred, prefix=""):
    mae_val = mean_absolute_error(y_true, y_pred)
    rmse_val = rmse(y_true, y_pred)
    mape_val = mape(y_true, y_pred)
    print(f"{prefix}MAE:  {mae_val:.4f}")
    print(f"{prefix}RMSE: {rmse_val:.4f}")
    print(f"{prefix}MAPE: {mape_val:.2f}%")
    return {'MAE': mae_val, 'RMSE': rmse_val, 'MAPE': mape_val}

In [6]:
# ==================== 时序块交叉验证 ====================

class TimeSeriesBlockCV:
    """
    时序块交叉验证
    验证集为连续块，训练集为剩余部分（最多两块）
    """
    
    def __init__(self, n_splits=5):
        self.n_splits = n_splits
    
    def split(self, X):
        n_samples = len(X)
        fold_size = n_samples // self.n_splits
        
        for i in range(self.n_splits):
            val_start = i * fold_size
            val_end = (i + 1) * fold_size if i < self.n_splits - 1 else n_samples
            
            val_idx = np.arange(val_start, val_end)
            train_idx = np.concatenate([
                np.arange(0, val_start),
                np.arange(val_end, n_samples)
            ])
            
            yield train_idx, val_idx
    
    def get_n_splits(self):
        return self.n_splits

In [7]:
# ==================== Optuna优化版本 ====================

def catboost_optuna_cv(X_train, y_train, X_test, y_test, cat_indices=None,
                        n_trials=30, cv_folds=5, random_state=42, use_gpu=False):
    """
    CatBoost Optuna + 时序块交叉验证 + 测试集评估
    
    参数:
    ------
    X_train, y_train : 训练数据
    X_test, y_test : 测试数据
    cat_indices : list
        类别特征索引，如 [8, 9, 10, 11, 12, 13, 14]
    n_trials : int
        优化迭代次数
    cv_folds : int
        交叉验证折数
    use_gpu : bool
        是否使用GPU
    """
    
    # 确保是numpy数组
    X_train = np.array(X_train, dtype=object)
    X_test = np.array(X_test, dtype=object)
    y_train = np.array(y_train)
    y_test = np.array(y_test)
    
    # 类别特征索引
    cat_features = cat_indices if cat_indices else []
    
    # 将类别特征转换为整数类型（CatBoost要求）
    if cat_features:
        for col in cat_features:
            X_train[:, col] = X_train[:, col].astype(int)
            X_test[:, col] = X_test[:, col].astype(int)
    
    print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
    print(f"类别特征索引: {cat_features}")
    print(f"优化次数: {n_trials}, 时序CV: {cv_folds}折, GPU: {use_gpu}")
    print("=" * 60)
    
    # 时序块交叉验证
    tscv = TimeSeriesBlockCV(n_splits=cv_folds)
    
    # 打印CV划分信息
    print("时序块划分:")
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        print(f"  Fold {fold+1}: train={len(train_idx)}, val={len(val_idx)} "
              f"[val范围: {val_idx[0]}-{val_idx[-1]}]")
    print("=" * 60)
    
    all_results = []
    
    def objective(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 200, 2000),
            'depth': trial.suggest_int('depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
            'border_count': trial.suggest_int('border_count', 32, 255),
        }
        
        fold_metrics = {'MAE': [], 'RMSE': [], 'MAPE': []}
        
        for train_idx, val_idx in tscv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
            # 创建Pool
            train_pool = CatBoostPool(X_tr, y_tr, cat_features=cat_features)
            val_pool = CatBoostPool(X_val, y_val, cat_features=cat_features)
            
            model = CatBoostRegressor(
                **params,
                loss_function='MAE',
                task_type='GPU' if use_gpu else 'CPU',
                random_seed=random_state,
                verbose=False,
                early_stopping_rounds=50,
                thread_count=1
            )
            
            model.fit(train_pool, eval_set=val_pool, verbose=False)
            
            y_pred = model.predict(X_val)
            fold_metrics['MAE'].append(mean_absolute_error(y_val, y_pred))
            fold_metrics['RMSE'].append(rmse(y_val, y_pred))
            fold_metrics['MAPE'].append(mape(y_val, y_pred))
        
        result = {
            'trial': trial.number + 1,
            'params': params,
            'MAE_mean': np.mean(fold_metrics['MAE']),
            'MAE_std': np.std(fold_metrics['MAE']),
            'RMSE_mean': np.mean(fold_metrics['RMSE']),
            'RMSE_std': np.std(fold_metrics['RMSE']),
            'MAPE_mean': np.mean(fold_metrics['MAPE']),
            'MAPE_std': np.std(fold_metrics['MAPE']),
        }
        all_results.append(result)
        
        print(f"Trial {trial.number+1:3d}: MAE={result['MAE_mean']:.4f}, "
              f"RMSE={result['RMSE_mean']:.4f}, MAPE={result['MAPE_mean']:.2f}%")
        
        return result['MAE_mean']
    
    # 优化
    study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    best_params = study.best_params
    
    # 最终模型
    print("\n" + "=" * 60)
    print("使用最佳参数训练最终模型...")
    
    train_pool_full = CatBoostPool(X_train, y_train, cat_features=cat_features)
    test_pool = CatBoostPool(X_test, y_test, cat_features=cat_features)
    
    best_model = CatBoostRegressor(
        **best_params,
        loss_function='MAE',
        task_type='GPU' if use_gpu else 'CPU',
        random_seed=random_state,
        verbose=False,
        thread_count=1
    )
    best_model.fit(train_pool_full, verbose=False)
    
    # 输出
    print("\n" + "=" * 60)
    print("最佳参数:")
    for k, v in best_params.items():
        print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")
    
    best_cv_result = min(all_results, key=lambda x: x['MAE_mean'])
    # print(f"\n【交叉验证性能】")
    print(f"  MAE:  {best_cv_result['MAE_mean']:.4f} ± {best_cv_result['MAE_std']:.4f}")
    print(f"  RMSE: {best_cv_result['RMSE_mean']:.4f} ± {best_cv_result['RMSE_std']:.4f}")
    print(f"  MAPE: {best_cv_result['MAPE_mean']:.2f}% ± {best_cv_result['MAPE_std']:.2f}%")
    
    # 测试集评估
    print(f"\n【测试集性能】")
    y_pred_test = best_model.predict(X_test)
    test_metrics = evaluate(y_test, y_pred_test, prefix="  ")
    
    return best_model, best_params, all_results, test_metrics, y_pred_test

In [19]:
# ==================== 保存模型 ====================

def save_model(model, best_params, test_metrics, model_name, save_dir='./models'):
    """
    保存模型、参数和指标
    
    参数:
        model: 训练好的模型
        best_params: 最佳参数字典
        test_metrics: 测试指标字典
        model_name: 模型名称 ('xgboost', 'lightgbm', 'catboost')
        save_dir: 保存目录
    """
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    # 保存模型
    if model_name == 'xgboost':
        model.save_model(f'{save_dir}/{model_name}_model.json')
    elif model_name == 'lightgbm':
        model.save_model(f'{save_dir}/{model_name}_model.txt')
    elif model_name == 'catboost':
        model.save_model(f'{save_dir}/{model_name}_model.cbm')
    
    # 保存参数和指标
    info = {'best_params': best_params, 'test_metrics': test_metrics}
    with open(f'{save_dir}/{model_name}_info.json', 'w') as f:
        json.dump(info, f, indent=2)
    
    print(f"模型已保存到 {save_dir}/{model_name}_model.*")

In [13]:
# 单次训练
X_2d = np.load(f'./train_data/TD_X_2d_ERCOT.npy')
y = np.load(f'./train_data/TD_y_ERCOT.npy')

print("划分数据集...")
test_size = 24 * 365 * 1  # 最后2年作为测试集

X_train, X_test = X_2d[:-test_size], X_2d[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]

best_model, best_params, cv_results, test_metrics, y_pred = catboost_optuna_cv(
    X_train, y_train, X_test, y_test,
    cat_indices=[8, 9, 10, 11, 12, 13, 14],
    n_trials=20,
    cv_folds=3,
    use_gpu=False
)
# save_model(best_model, best_params, test_metrics, v,  "catboost")

划分数据集...
训练集: (69912, 21), 测试集: (8760, 21)
类别特征索引: [8, 9, 10, 11, 12, 13, 14]
优化次数: 20, 时序CV: 3折, GPU: False
时序块划分:
  Fold 1: train=46608, val=23304 [val范围: 0-23303]
  Fold 2: train=46608, val=23304 [val范围: 23304-46607]
  Fold 3: train=46608, val=23304 [val范围: 46608-69911]


  0%|          | 0/20 [00:00<?, ?it/s]

Trial   1: MAE=2293.2590, RMSE=3276.4738, MAPE=5.15%
Trial   2: MAE=2170.9723, RMSE=3147.7034, MAPE=4.87%
Trial   3: MAE=2130.4983, RMSE=3089.4271, MAPE=4.80%
Trial   4: MAE=2126.6566, RMSE=3088.7113, MAPE=4.79%
Trial   5: MAE=2106.9980, RMSE=3061.5400, MAPE=4.74%
Trial   6: MAE=2113.9515, RMSE=3079.9274, MAPE=4.76%
Trial   7: MAE=2253.6589, RMSE=3218.2850, MAPE=5.04%
Trial   8: MAE=2224.4526, RMSE=3186.5705, MAPE=4.99%
Trial   9: MAE=2429.7802, RMSE=3464.8015, MAPE=5.46%
Trial  10: MAE=2107.8008, RMSE=3075.6822, MAPE=4.74%
Trial  11: MAE=2169.6611, RMSE=3115.7786, MAPE=4.88%
Trial  12: MAE=2096.9338, RMSE=3057.0352, MAPE=4.72%
Trial  13: MAE=2104.6174, RMSE=3065.3627, MAPE=4.74%
Trial  14: MAE=2100.8044, RMSE=3057.2975, MAPE=4.74%
Trial  15: MAE=2185.6009, RMSE=3154.0762, MAPE=4.91%
Trial  16: MAE=2121.7762, RMSE=3078.7282, MAPE=4.78%
Trial  17: MAE=2097.7523, RMSE=3061.2239, MAPE=4.72%
Trial  18: MAE=2111.8634, RMSE=3077.1597, MAPE=4.75%
Trial  19: MAE=2124.1299, RMSE=3095.6462, MAPE

In [ ]:
# import time
# import multiprocessing as mp
# from multiprocessing import Pool as ProcessPool, Manager

# def process_single_region_with_progress(args):
#     """处理单个区域的模型训练（带进度共享）"""
#     name, cat_indices, n_trials, cv_folds, use_gpu, progress_dict = args
    
#     try:
#         progress_dict[name] = "加载数据..."
        
#         # 加载数据
#         X_2d = np.load(f'./train_data/TD_X_2d_{name}.npy')
#         y = np.load(f'./train_data/TD_y_{name}.npy')
        
#         progress_dict[name] = "划分数据集..."
        
#         # 划分数据集
#         test_size = 24 * 365 * 2
#         X_train, X_test = X_2d[:-test_size], X_2d[-test_size:]
#         y_train, y_test = y[:-test_size], y[-test_size:]
        
#         progress_dict[name] = "训练模型..."
        
#         # 训练模型
#         best_model, best_params, cv_results, test_metrics, y_pred = catboost_optuna_cv(
#             X_train, y_train, X_test, y_test,
#             cat_indices=cat_indices,
#             n_trials=n_trials,
#             cv_folds=cv_folds,
#             use_gpu=use_gpu
#         )
        
#         progress_dict[name] = "保存模型..."
        
#         # 保存模型
#         save_model(best_model, best_params, test_metrics, name, "catboost")
        
#         progress_dict[name] = "完成"
#         return (True, name, test_metrics)
    
#     except Exception as e:
#         progress_dict[name] = f"错误: {str(e)}"
#         return (False, name, str(e))


# def monitor_progress(progress_dict, names, interval=5):
#     """监控进度的辅助函数"""
#     while True:
#         print("\n--- 当前进度 ---")
#         completed = 0
#         for name in names:
#             status = progress_dict.get(name, "等待中...")
#             print(f"  {name}: {status}")
#             if status == "完成" or status.startswith("错误"):
#                 completed += 1
        
#         if completed == len(names):
#             break
#         time.sleep(interval)


# def run_multiprocess_training_v2(names, cat_indices, n_trials=30, cv_folds=5, use_gpu=False):
#     """多进程运行训练（增强版）"""
    
#     num_processes = min(len(names), mp.cpu_count())
#     print(f"使用 {num_processes} 个进程进行训练...")
#     print(f"待处理区域: {names}")
    
#     # 使用 Manager 创建共享字典
#     manager = Manager()
#     progress_dict = manager.dict()
    
#     # 准备参数
#     args_list = [
#         (name, cat_indices, n_trials, cv_folds, use_gpu, progress_dict) 
#         for name in names
#     ]
    
#     # 启动进程池
#     with ProcessPool(processes=num_processes) as pool:
#         # 异步执行
#         async_result = pool.map_async(process_single_region_with_progress, args_list)
        
#         # 等待完成，同时可以监控进度
#         while not async_result.ready():
#             # 打印当前状态
#             status_line = " | ".join([f"{n}:{progress_dict.get(n, '等待')[:6]}" for n in names])
#             print(f"\r{status_line}", end="", flush=True)
#             time.sleep(2)
        
#         results = async_result.get()
    
#     print()  # 换行
#     return results


# if __name__ == '__main__':
#     names = ['COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH', 'SCENT', 'WEST', 'ERCOT']
    
#     results = run_multiprocess_training_v2(
#         names=names,
#         cat_indices=[8, 9, 10, 11, 12, 13, 14],
#         n_trials=30,
#         cv_folds=5,
#         use_gpu=False
#     )
    
#     # 结果汇总
#     print("\n" + "="*60)
#     print("训练结果汇总:")
#     print("="*60)
    
#     for success, name, info in results:
#         status = "✓ 成功" if success else "✗ 失败"
#         print(f"{status} | {name:8s} | {info}")

In [ ]:
# # ==================== 快速网格搜索版本 ====================

# def catboost_fast_cv(X_train, y_train, X_test, y_test, cat_indices=None,
#                      cv_folds=5, random_state=42, use_gpu=False):
#     """快速网格搜索 + 时序块交叉验证"""
#     from itertools import product
    
#     X_train = np.array(X_train, dtype=object)
#     X_test = np.array(X_test, dtype=object)
#     y_train = np.array(y_train)
#     y_test = np.array(y_test)
    
#     cat_features = cat_indices if cat_indices else []
    
#     # 将类别特征转换为整数类型
#     if cat_features:
#         for col in cat_features:
#             X_train[:, col] = X_train[:, col].astype(int)
#             X_test[:, col] = X_test[:, col].astype(int)
    
#     param_grid = {
#         'iterations': [200, 400, 600, 800, 1000],
#         'depth': [4, 6, 8, 10],
#         'learning_rate': [0.05, 0.1],
#         'l2_leaf_reg': [1, 5],
#     }
    
#     param_names = list(param_grid.keys())
#     param_combinations = list(product(*param_grid.values()))
#     total = len(param_combinations)
    
#     print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
#     print(f"参数组合: {total}, 时序CV: {cv_folds}折")
#     print("=" * 60)
    
#     tscv = TimeSeriesBlockCV(n_splits=cv_folds)
    
#     # 打印划分
#     print("时序块划分:")
#     for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
#         print(f"  Fold {fold+1}: train={len(train_idx)}, val={len(val_idx)} "
#               f"[val范围: {val_idx[0]}-{val_idx[-1]}]")
#     print("=" * 60)
    
#     all_results = []
#     best_mae = float('inf')
#     best_params = None
    
#     for idx, params in enumerate(param_combinations):
#         current_params = dict(zip(param_names, params))
#         fold_metrics = {'MAE': [], 'RMSE': [], 'MAPE': []}
        
#         for train_idx, val_idx in tscv.split(X_train):
#             X_tr, X_val = X_train[train_idx], X_train[val_idx]
#             y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
#             train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
#             val_pool = Pool(X_val, y_val, cat_features=cat_features)
            
#             model = CatBoostRegressor(
#                 **current_params,
#                 loss_function='MAE',
#                 task_type='GPU' if use_gpu else 'CPU',
#                 random_seed=random_state,
#                 verbose=False,
#                 early_stopping_rounds=50
#             )
            
#             model.fit(train_pool, eval_set=val_pool, verbose=False)
            
#             y_pred = model.predict(X_val)
#             fold_metrics['MAE'].append(mean_absolute_error(y_val, y_pred))
#             fold_metrics['RMSE'].append(rmse(y_val, y_pred))
#             fold_metrics['MAPE'].append(mape(y_val, y_pred))
        
#         result = {
#             'params': current_params,
#             'MAE_mean': np.mean(fold_metrics['MAE']),
#             'MAE_std': np.std(fold_metrics['MAE']),
#             'RMSE_mean': np.mean(fold_metrics['RMSE']),
#             'RMSE_std': np.std(fold_metrics['RMSE']),
#             'MAPE_mean': np.mean(fold_metrics['MAPE']),
#             'MAPE_std': np.std(fold_metrics['MAPE']),
#         }
#         all_results.append(result)
        
#         print(f"[{idx+1}/{total}] {current_params} → MAE={result['MAE_mean']:.4f}")
        
#         if result['MAE_mean'] < best_mae:
#             best_mae = result['MAE_mean']
#             best_params = current_params
    
#     # 最终模型
#     print("\n" + "=" * 60)
#     train_pool_full = Pool(X_train, y_train, cat_features=cat_features)
    
#     best_model = CatBoostRegressor(
#         **best_params,
#         loss_function='MAE',
#         task_type='GPU' if use_gpu else 'CPU',
#         random_seed=random_state,
#         verbose=False
#     )
#     best_model.fit(train_pool_full, verbose=False)
    
#     print(f"最佳参数: {best_params}")
    
#     best_cv_result = min(all_results, key=lambda x: x['MAE_mean'])
#     print(f"\n【交叉验证性能】")
#     print(f"  MAE:  {best_cv_result['MAE_mean']:.4f} ± {best_cv_result['MAE_std']:.4f}")
#     print(f"  RMSE: {best_cv_result['RMSE_mean']:.4f} ± {best_cv_result['RMSE_std']:.4f}")
#     print(f"  MAPE: {best_cv_result['MAPE_mean']:.2f}% ± {best_cv_result['MAPE_std']:.2f}%")
    
#     # 测试集
#     print(f"\n【测试集性能】")
#     y_pred_test = best_model.predict(X_test)
#     test_metrics = evaluate(y_test, y_pred_test, prefix="  ")
    
#     return best_model, best_params, all_results, test_metrics, y_pred_test

In [ ]:
# best_model, best_params, cv_results, test_metrics, y_pred = catboost_fast_cv(
#     X_train, y_train, X_test, y_test,
#     cat_indices=[8, 9, 10, 11, 12, 13, 14],
#     cv_folds=5,
#     use_gpu=True
# )

In [ ]:
# ==================== 读取模型 ====================

def load_model(model_name, save_dir='./models'):
    """
    读取模型、参数和指标
    
    返回:
        model, best_params, test_metrics
    """
    import xgboost as xgb
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    
    # 读取模型
    if model_name == 'xgboost':
        model = xgb.Booster()
        model.load_model(f'{save_dir}/{model_name}_model.json')
    elif model_name == 'lightgbm':
        model = lgb.Booster(model_file=f'{save_dir}/{model_name}_model.txt')
    elif model_name == 'catboost':
        model = CatBoostRegressor()
        model.load_model(f'{save_dir}/{model_name}_model.cbm')
    
    # 读取参数和指标
    with open(f'{save_dir}/{model_name}_info.json', 'r') as f:
        info = json.load(f)
    
    print(f"模型已从 {save_dir}/{model_name}_model.* 加载")
    return model, info['best_params'], info['test_metrics']

In [20]:
save_model(best_model, best_params, test_metrics, "catboost")

模型已保存到 ./models/catboost_model.*


In [8]:
model, params, metrics = load_model('catboost')

模型已从 ./models/catboost_model.* 加载


In [11]:
model1, params, metrics = load_model('catboost')
model2, params, metrics = load_model('xgboost')

模型已从 ./models/catboost_model.* 加载
模型已从 ./models/xgboost_model.* 加载


In [74]:
# 确保是numpy数组
X_test = np.array(X_test, dtype=object)
y_test = np.array(y_test)

# 类别特征索引
cat_indices=[8, 9, 10, 11, 12, 13, 14]
cat_features = cat_indices if cat_indices else []

# 将类别特征转换为整数类型（CatBoost要求）

for col in cat_features:
    X_test[:, col] = X_test[:, col].astype(int)

In [75]:
y1 = model1.predict(X_test) / 2

In [64]:
import xgboost as xgb

In [72]:
# 确保是numpy数组
y_test = np.array(y_test)

# 转为DataFrame并设置类别特征
X_test_df = pd.DataFrame(X_test)
for col in cat_indices:
    X_test_df[col] = X_test_df[col].astype('category')

In [73]:
X_test_df = xgb.DMatrix(X_test_df, np.array([1] * X_test_df.shape[0]).reshape(-1, 1), enable_categorical=True)
y2 = model2.predict(X_test_df) / 2

In [57]:
model3, params, metrics = load_model('lightgbm')

模型已从 ./models/lightgbm_model.* 加载


In [60]:
# 确保是numpy数组
X_test = np.array(X_test)
y_test = np.array(y_test)

if cat_features:
    X_test = X_test.copy()
    for col in cat_features:
        X_test[:, col] = X_test[:, col].astype(int)

In [61]:
y3 = model3.predict(X_test) / 3

In [76]:
evaluate(y_test, y1+y2)

MAE:  2249.5332
RMSE: 3242.7673
MAPE: 4.23%


{'MAE': 2249.5331744849273,
 'RMSE': 3242.7672609859337,
 'MAPE': 4.230326983589572}